In [308]:
!pip install kagglehub

In [309]:
import numpy as np
import pandas as pd

In [310]:
movies = pd.read_csv('tmdb_5000_movies.csv')
credits_movies = pd.read_csv('tmdb_5000_credits.csv')

In [311]:
print(movies.shape) 
print(credits_movies.shape)

(4803, 20)
(4803, 4)


In [312]:
movies.head(2)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500


In [313]:
credits_movies.head(2)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


Goal: recommender system which recommends using tags
we have to remove unnecessary column that doesn't provide us with tags 

# Preprocessing

In [314]:
print(movies.columns.values)

['budget' 'genres' 'homepage' 'id' 'keywords' 'original_language'
 'original_title' 'overview' 'popularity' 'production_companies'
 'production_countries' 'release_date' 'revenue' 'runtime'
 'spoken_languages' 'status' 'tagline' 'title' 'vote_average' 'vote_count']


In [315]:
movies.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800


In [316]:
movies = movies[[
# 'budget' people doesn't search movies based on budget not required
'genres',
# 'homepage'
 'id' ,'keywords',
 # 'original_language'
 'original_title' ,
 'overview' 
 # 'popularity' 'production_companies'
#  'production_countries' 'release_date' 'revenue' 'runtime'
#  'spoken_languages' 'status' 'tagline' 'title' 'vote_average' 'vote_count'
]]

In [317]:
print(credits_movies.columns.values)

['movie_id' 'title' 'cast' 'crew']


In [318]:
credits_movies.head(1)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [319]:
credits_movies = credits_movies[[
    'movie_id', 
    # 'title' 
    'cast', 'crew'
]]

In [320]:
# we don't need two tables to build this recommender system we can work with one table
movies = movies.merge(credits_movies, left_on='id',right_on='movie_id')


In [321]:
movies = movies.drop("movie_id", axis=1)

In [322]:
# first need to remove duplicate and no avail data
movies.isna().sum()

genres            0
id                0
keywords          0
original_title    0
overview          3
cast              0
crew              0
dtype: int64

In [323]:
movies.dropna(inplace=True)

In [324]:
movies.duplicated().sum()

np.int64(0)

In [325]:
movies.iloc[0] # checkout a single data

genres            [{"id": 28, "name": "Action"}, {"id": 12, "nam...
id                                                            19995
keywords          [{"id": 1463, "name": "culture clash"}, {"id":...
original_title                                               Avatar
overview          In the 22nd century, a paraplegic Marine is di...
cast              [{"cast_id": 242, "character": "Jake Sully", "...
crew              [{"credit_id": "52fe48009251416c750aca23", "de...
Name: 0, dtype: object

We only care about words or keywords so we have to pre-process and convert all the object and get their keywords
generes we extract their name and keep it in a list
keyword we extract the name keys value
in cast we take the top 3 cast
crew we take the name of director

In [326]:
import ast
def extractGenres(object):
    name = []
    for item in ast.literal_eval(object):
        name.append(item['name'])
    return name

movies['genres'] = movies['genres'].apply(extractGenres) # go to each genres and extract the geners
movies['keywords'] = movies['keywords'].apply(extractGenres)

In [327]:
def extractCast(object, counter=3):
    name = []
    count = 0
    for item in ast.literal_eval(object):
        if count == counter:
            break
        name.append(item['name'])
        count += 1
    return name
movies['cast'] = movies['cast'].apply(extractCast)

In [328]:
def extractCast(object, counter=3):
    name = []
  
    for item in ast.literal_eval(object):
        if item['job'] == 'Director':
            name.append(item['name'])
    return name

movies['crew'] = movies['crew'].apply(extractCast)

In [329]:
# join all the list and create a single tag as string
# in cast and director name we have to change sam worhtington -> samWorhtington so that sam Worhtington and sam mendes are different
# they can be recommended sam because of sam in their name

In [330]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ", "") for i in x])
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ", "") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ", "") for i in x])
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ", "") for i in x])
movies

,genres,id,keywords,original_title,overview,cast,crew
0,"[Action, Adventure, Fantasy, ScienceFiction]",19995,"[cultureclash, future, spacewar, spacecolony, ...",Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron]
1,"[Adventure, Fantasy, Action]",285,"[ocean, drugabuse, exoticisland, eastindiatrad...",Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski]
2,"[Action, Adventure, Crime]",206647,"[spy, basedonnovel, secretagent, sequel, mi6, ...",Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[DanielCraig, ChristophWaltz, LéaSeydoux]",[SamMendes]
3,"[Action, Crime, Drama, Thriller]",49026,"[dccomics, crimefighter, terrorist, secretiden...",The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[ChristianBale, MichaelCaine, GaryOldman]",[ChristopherNolan]
4,"[Action, Adventure, ScienceFiction]",49529,"[basedonnovel, mars, medallion, spacetravel, p...",John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[TaylorKitsch, LynnCollins, SamanthaMorton]",[AndrewStanton]
...,...,...,...,...,...,...,...
4798,"[Action, Crime, Thriller]",9367,"[unitedstates–mexicobarrier, legs, arms, paper...",El Mariachi,"[El, Mariachi, just, wants, to, play, his, gui...","[CarlosGallardo, JaimedeHoyos, PeterMarquardt]",[RobertRodriguez]
4799,"[Comedy, Romance]",72766,[],Newlyweds,"[A, newlywed, couple's, honeymoon, is, upended...","[EdwardBurns, KerryBishé, MarshaDietlein]",[EdwardBurns]
4800,"[Comedy, Drama, Romance, TVMovie]",231617,"[date, loveatfirstsight, narration, investigat...","Signed, Sealed, Delivered","[""Signed,, Sealed,, Delivered"", introduces, a,...","[EricMabius, KristinBooth, CrystalLowe]",[ScottSmith]
4801,[],126186,[],Shanghai Calling,"[When, ambitious, New, York, attorney, Sam, is...","[DanielHenney, ElizaCoupe, BillPaxton]",[DanielHsia]


In [331]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords']  + movies['cast']  + movies['crew']
movies

,genres,id,keywords,original_title,overview,cast,crew,tags
0,"[Action, Adventure, Fantasy, ScienceFiction]",19995,"[cultureclash, future, spacewar, spacecolony, ...",Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron],"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,"[Adventure, Fantasy, Action]",285,"[ocean, drugabuse, exoticisland, eastindiatrad...",Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski],"[Captain, Barbossa,, long, believed, to, be, d..."
2,"[Action, Adventure, Crime]",206647,"[spy, basedonnovel, secretagent, sequel, mi6, ...",Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[DanielCraig, ChristophWaltz, LéaSeydoux]",[SamMendes],"[A, cryptic, message, from, Bond’s, past, send..."
3,"[Action, Crime, Drama, Thriller]",49026,"[dccomics, crimefighter, terrorist, secretiden...",The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[ChristianBale, MichaelCaine, GaryOldman]",[ChristopherNolan],"[Following, the, death, of, District, Attorney..."
4,"[Action, Adventure, ScienceFiction]",49529,"[basedonnovel, mars, medallion, spacetravel, p...",John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[TaylorKitsch, LynnCollins, SamanthaMorton]",[AndrewStanton],"[John, Carter, is, a, war-weary,, former, mili..."
...,...,...,...,...,...,...,...,...
4798,"[Action, Crime, Thriller]",9367,"[unitedstates–mexicobarrier, legs, arms, paper...",El Mariachi,"[El, Mariachi, just, wants, to, play, his, gui...","[CarlosGallardo, JaimedeHoyos, PeterMarquardt]",[RobertRodriguez],"[El, Mariachi, just, wants, to, play, his, gui..."
4799,"[Comedy, Romance]",72766,[],Newlyweds,"[A, newlywed, couple's, honeymoon, is, upended...","[EdwardBurns, KerryBishé, MarshaDietlein]",[EdwardBurns],"[A, newlywed, couple's, honeymoon, is, upended..."
4800,"[Comedy, Drama, Romance, TVMovie]",231617,"[date, loveatfirstsight, narration, investigat...","Signed, Sealed, Delivered","[""Signed,, Sealed,, Delivered"", introduces, a,...","[EricMabius, KristinBooth, CrystalLowe]",[ScottSmith],"[""Signed,, Sealed,, Delivered"", introduces, a,..."
4801,[],126186,[],Shanghai Calling,"[When, ambitious, New, York, attorney, Sam, is...","[DanielHenney, ElizaCoupe, BillPaxton]",[DanielHsia],"[When, ambitious, New, York, attorney, Sam, is..."


In [332]:
movies_clean = movies[['id', 'original_title' , 'tags']]
movies_clean

,id,original_title,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili..."
...,...,...,...
4798,9367,El Mariachi,"[El, Mariachi, just, wants, to, play, his, gui..."
4799,72766,Newlyweds,"[A, newlywed, couple's, honeymoon, is, upended..."
4800,231617,"Signed, Sealed, Delivered","[""Signed,, Sealed,, Delivered"", introduces, a,..."
4801,126186,Shanghai Calling,"[When, ambitious, New, York, attorney, Sam, is..."


In [333]:
movies_clean['tags'] = movies_clean['tags'].apply(lambda x:" ".join(x))
movies_clean['tags'] = movies_clean['tags'].apply(lambda x:x.lower())

/tmp/ipykernel_32297/596467669.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movies_clean['tags'] = movies_clean['tags'].apply(lambda x:" ".join(x))
/tmp/ipykernel_32297/596467669.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movies_clean['tags'] = movies_clean['tags'].apply(lambda x:x.lower())


In [334]:
movies_clean

,id,original_title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."
...,...,...,...
4798,9367,El Mariachi,el mariachi just wants to play his guitar and ...
4799,72766,Newlyweds,a newlywed couple's honeymoon is upended by th...
4800,231617,"Signed, Sealed, Delivered","""signed, sealed, delivered"" introduces a dedic..."
4801,126186,Shanghai Calling,when ambitious new york attorney sam is sent t...


# vectorization

to make a recommender system we just have to calculate similarity score.
- first have to convert all the text to vector
  -  remove stem words 
  -  remove stop words
  -  convert to bow
- calcuate similarity

# remove punctuation and stop words

In [351]:
from nltk.corpus import stopwords
import re

stop_words = set(stopwords.words('english'))

def remove_stop_words(text):
    text = re.sub(r'[^\w\s]', '', text)
    y = [x for x in text.split() if not x in stop_words]
    return " ".join(y)

movies_clean['tags'] = movies_clean['tags'].apply(remove_stop_words)
movies_clean.iloc[0]['tags']

/tmp/ipykernel_32297/1521161526.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movies_clean['tags'] = movies_clean['tags'].apply(remove_stop_words)


'22nd century paraplegic marine dispatched moon pandora unique mission becomes torn following orders protecting alien civilization action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d samworthington zoesaldana sigourneyweaver jamescameron'

# remove stem words
[loving, love, loved] -> [love, love, love] because all of them have same meaning

In [ ]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()
def steming(text):
    y = []
    for word in text.split(" "):
        y.append(stemmer.stem(word))
    return " ".join(y)


movies_clean['tags'] = movies_clean['tags'].apply(steming)


/tmp/ipykernel_32297/766502836.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movies_clean['tags'] = movies_clean['tags'].apply(steming)


# convert to bow

In [356]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=5000)
vectors = cv.fit_transform(movies_clean['tags']).toarray()

# calculate similarity

In [363]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vectors)

In [364]:
print(similarity)

[[1.         0.08134892 0.08017837 ... 0.06228411 0.02383656 0.        ]
 [0.08134892 1.         0.08696566 ... 0.02251887 0.         0.        ]
 [0.08017837 0.08696566 1.         ... 0.04438968 0.         0.        ]
 ...
 [0.06228411 0.02251887 0.04438968 ... 1.         0.03959038 0.01979519]
 [0.02383656 0.         0.         ... 0.03959038 1.         0.02272727]
 [0.         0.         0.         ... 0.01979519 0.02272727 1.        ]]


# recommend movies

In [391]:
def recommend(movie):
    movie_index = movies_clean[movies_clean['original_title'] == movie].index[0]
    distance = similarity[movie_index]
    movie_list = sorted(list(enumerate(distance)), reverse=True, key=lambda x:x[1])[1:6]
    return movie_list
    
for i in recommend('Batman Begins'):
    print(movies_clean.iloc[i[0]]['original_title'])

The Dark Knight
The Dark Knight Rises
Batman v Superman: Dawn of Justice
Batman
Batman & Robin


# save it in a pickle file

In [392]:
import pickle

pickle.dump(movies_clean, open('movies.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))